# Load datasets
This step loads the two datasets: GoEmotions (kaggle) and Political tweets(huggingface).
The datasets are cleaned, the GoEmotions dataset emotions are reduced to 10, and finally the datasets are combined into one clean dataset. 

In [1]:
!pip install datasets

In [2]:
import pandas as pd

url = "https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_1.csv"
goemotions = pd.read_csv(url)

print(goemotions.head())
print(goemotions.columns)

                                                text       id  \
0                                    That game hurt.  eew5j0j   
1   >sexuality shouldn’t be a grouping category I...  eemcysk   
2     You do right, if you don't care then fuck 'em!  ed2mah1   
3                                 Man I love reddit.  eeibobj   
4  [NAME] was nowhere near them, he was by the Fa...  eda6yn6   

                author            subreddit    link_id   parent_id  \
0                Brdd9                  nrl  t3_ajis4z  t1_eew18eq   
1          TheGreen888     unpopularopinion  t3_ai4q37   t3_ai4q37   
2             Labalool          confessions  t3_abru74  t1_ed2m7g7   
3        MrsRobertshaw             facepalm  t3_ahulml   t3_ahulml   
4  American_Fascist713  starwarsspeculation  t3_ackt2f  t1_eda65q2   

    created_utc  rater_id  example_very_unclear  admiration  ...  love  \
0  1.548381e+09         1                 False           0  ...     0   
1  1.548084e+09        37               

In [3]:
# political dataset from huggingface download + convert to df
from datasets import load_dataset

political = load_dataset("tweet_eval", "sentiment")
print(political)
train_df = political["train"].to_pandas()
test_df = political["test"].to_pandas()

# export file
train_df.to_csv("tweet_eval_sentiment.csv", index=False)

tweets = pd.concat([train_df, test_df], ignore_index=True)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [4]:
# rename columns
tweets = tweets.rename(columns={"text": "text", "label": "label"})

In [5]:
import re

def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+", "", text)   # remove links
    text = re.sub(r"@\w+", "", text)      # remove mentions
    text = re.sub(r"#\w+", "", text)      # remove hashtags
    text = re.sub(r"[^a-z\s]", "", text)  # remove special chars
    text = re.sub(r"\s+", " ", text).strip()
    return text

goemotions["text"] = goemotions["text"].apply(clean_text)
tweets["text"] = tweets["text"].apply(clean_text)

In [6]:
goemotions.head()

,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,that game hurt,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,sexuality shouldnt be a grouping category it m...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,you do right if you dont care then fuck em,ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1
3,man i love reddit,eeibobj,MrsRobertshaw,facepalm,t3_ahulml,t3_ahulml,1.547965e+09,18,False,0,...,1,0,0,0,0,0,0,0,0,0
4,name was nowhere near them he was by the falcon,eda6yn6,American_Fascist713,starwarsspeculation,t3_ackt2f,t1_eda65q2,1.546669e+09,2,False,0,...,0,0,0,0,0,0,0,0,0,1


In [7]:
tweets.head()

,text,label
0,qt in the original draft of the th book remus ...,2
1,ben smith smith concussion remains out of the ...,1
2,sorry bout the stream last night i crashed out...,1
3,chase headleys rbi double in the th inning off...,1
4,alciato bee will invest million in january ano...,2


In [ ]:
# clean text
import re

def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+", "", text)   # remove links
    text = re.sub(r"@\w+", "", text)      # remove mentions
    text = re.sub(r"#\w+", "", text)      # remove hashtags
    text = re.sub(r"[^a-z\s]", "", text)  # remove special chars
    text = re.sub(r"\s+", " ", text).strip()
    return text

goemotions["text"] = goemotions["text"].apply(clean_text)
tweets["text"] = tweets["text"].apply(clean_text)

In [ ]:
#reduce to 10 emotions
emotion_map = {
    "joy": ["joy", "amusement", "excitement"],
    "love": ["love", "caring", "admiration"],
    "optimism": ["optimism", "desire", "approval"],
    "gratitude": ["gratitude", "relief", "pride"],
    
    "anger": ["anger", "annoyance", "disapproval"],
    "disgust": ["disgust"],
    "fear": ["fear", "nervousness"],
    "sadness": ["sadness", "grief", "disappointment", "remorse"],
    "embarrassment": ["embarrassment"],
    
    "neutral": ["neutral", "confusion", "curiosity", "realization", "surprise"]
}

In [ ]:
#convert goemotions to a single label
def map_emotions(row):
    for new_label, old_labels in emotion_map.items():
        for label in old_labels:
            if label in row and row[label] == 1:
                return new_label
    return "neutral"

goemotions["label"] = goemotions.apply(map_emotions, axis=1)

goemotions = goemotions[["text", "label"]]

In [ ]:
# clean political tweets labels
def map_tweet(label):
    if label == 0:
        return "anger"
    elif label == 1:
        return "neutral"
    else:
        return "joy"

tweets["label"] = tweets["label"].apply(map_tweet)
tweets = tweets[["text", "label"]]

In [ ]:
# combine datasets
df = pd.concat([goemotions, tweets], ignore_index=True)

print(df["label"].value_counts())

In [ ]:
id2label = {v:k for k,v in label_map.items()}

print(id2label)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# keep readable emotion labels
df["emotion_name"] = df["label"]

# encode numeric labels for BERT
le = LabelEncoder()
df["label"] = le.fit_transform(df["emotion_name"])

# mapping dictionary
label_map = dict(zip(le.classes_, le.transform(le.classes_)))

print(label_map)

In [ ]:
# final dataset
print(df.head())
df.to_csv("cleaned_emotion_dataset.csv", index=False)

print("Dataset saved successfully")

In [ ]:
# convert to hugging face dataset for BERT
from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1)

# Tokenization

In [ ]:
import sys
!{sys.executable} -m pip install transformers[torch]

In [ ]:
# tokenizer - hugging face transformer
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# apply tokenization
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

dataset = dataset.map(tokenize, batched=True)

#set format for PyTorch
dataset = dataset.remove_columns(["text"])
dataset.set_format("torch")

In [ ]:
!pip install --upgrade transformers

In [ ]:
import transformers
print(transformers)

In [ ]:
# load BERT model
from transformers import BertForSequenceClassification

num_labels = len(set(df["label"]))

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)

print("Model loaded successfully")

In [ ]:
# training setup
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert-emotion-model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs"
)

In [ ]:
# trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

print("Trainer created successfully")

In [ ]:
# train model
trainer.train()

# evaluate
trainer.evaluate()

In [ ]:
# add metrics (Accuracy + F1)
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

# update trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics
)

In [ ]:
# retrain
trainer.train()

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
# generate predictions
predictions = trainer.predict(dataset["test"])

In [ ]:
# convert predictions to labels
preds = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

In [ ]:
# create confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(true_labels, preds)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
print(classification_report(true_labels, preds))

In [ ]:
# saved trained model
trainer.save_model("./final-emotion-model")
tokenizer.save_pretrained("./final-emotion-model")

In [ ]:
# test custom predictions
text = "I am really excited for the election results!"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

outputs = model(**inputs)

predicted_class = outputs.logits.argmax().item()

print("Predicted label:", predicted_class)

In [ ]:
# emotion names instead of numeric labels
id2label = {v:k for k,v in label_map.items()}

print("Predicted emotion:", id2label[predicted_class])